# MetaPilot Phase 2: QLoRA fine-tuning of the LLM selector (Colab T4)

Fine-tunes **Qwen2.5-1.5B-Instruct** with 4-bit QLoRA to rank the pipeline portfolio from a text profile of a
dataset, and writes its rankings so `metapilot evaluate` can score them next to the Phase 1 meta-models.

Protocol (same as Phase 1):
* one adapter per **outer fold** and prompt kind, trained only on tasks outside that fold;
* each adapter predicts only its own held-out fold, so every task is predicted by a model that never saw it;
* three variants: `base_numeric` (no fine-tuning), `ft_numeric` (Prompt A, same information as LightGBM) and
  `ft_semantic` (Prompt B, plus column names and description).

Nothing here falls back to a default ranking: `evaluate` reports `llm_coverage`, the share of pipelines the model
actually named, next to every LLM row.

**Runtime:** set `Runtime > Change runtime type > T4 GPU`. Work is saved to Google Drive after every fold, so a
disconnect only loses the fold in progress; re-run the notebook to continue.

In [ ]:
# 1. Settings
REPO = "https://github.com/atharv-0509/triage"
BRANCH = "claude/starting-from-beginning-joh2ym"
CONFIG = "configs/pilot.yaml"     # pilot: built here in a few minutes. Full study: see step 3.
META = "data/pilot"               # meta-dataset directory inside the repo
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EPOCHS = 3
KINDS = ["numeric", "semantic"]
DRIVE_DIR = "/content/drive/MyDrive/metapilot_phase2"   # adapters + predictions are kept here

In [ ]:
# 2. Get the code and install (bitsandbytes is needed for 4-bit loading on the GPU)
import os, subprocess
if not os.path.exists("/content/triage"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, "/content/triage"], check=True)
%cd /content/triage
!pip install -q -e ".[llm]" bitsandbytes

import torch
assert torch.cuda.is_available(), "No GPU: set Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 3. Meta-dataset and prompts.
# Pilot: build it here (a few minutes). Full study: upload the assembled data/full folder
# (CSV and JSONL files, no parts/ needed) to DRIVE_DIR/data_full and set META = DRIVE_DIR + "/data_full".
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(f"{META}/performance.csv"):
    if META.startswith("data/pilot"):
        !python -m metapilot build --config {CONFIG} --out {META}
    else:
        raise FileNotFoundError(f"{META}/performance.csv not found; upload the meta-dataset first")

!python -m metapilot prompts --config {CONFIG} --meta {META}
PROMPTS = f"{META}/llm/prompts.jsonl"

import json
rows = [json.loads(l) for l in open(PROMPTS)]
folds = sorted({r["fold"] for r in rows})
print(len(rows), "tasks, folds", folds)
print(rows[0]["numeric"]); print("TARGET:", rows[0]["target"])

In [ ]:
# 4. Zero-shot base model (no fine-tuning), Prompt A, all tasks
PRED_DIR = f"{DRIVE_DIR}/predictions"
os.makedirs(PRED_DIR, exist_ok=True)
!python -m metapilot llm-predict --prompts {PROMPTS} --kind numeric --model {MODEL} --out {PRED_DIR}/base_numeric.jsonl

In [ ]:
# 5. Fine-tune one adapter per fold and prompt kind, predict its held-out fold
for kind in KINDS:
    for k in folds:
        adapter = f"{DRIVE_DIR}/adapters/{kind}_fold{k}"
        if not os.path.exists(f"{adapter}/train_info.json"):
            !python -m metapilot llm-train --prompts {PROMPTS} --fold {k} --kind {kind} --model {MODEL} --epochs {EPOCHS} --out {adapter}
        !python -m metapilot llm-predict --prompts {PROMPTS} --kind {kind} --fold {k} --model {MODEL} --adapter {adapter} --out {PRED_DIR}/ft_{kind}.jsonl

In [ ]:
# 6. Score all methods on the shared folds; LLM rows show llm_coverage
import shutil
os.makedirs(f"{META}/llm/predictions", exist_ok=True)
for f in os.listdir(PRED_DIR):
    shutil.copy(f"{PRED_DIR}/{f}", f"{META}/llm/predictions/{f}")
!python -m metapilot evaluate --config {CONFIG} --meta {META} --out runs/phase2
import pandas as pd
display(pd.read_csv("runs/phase2/summary.csv", index_col=0).round(3))
display(pd.read_csv("runs/phase2/summary_real.csv", index_col=0).round(3))   # RQ3: real datasets only
shutil.copytree("runs/phase2", f"{DRIVE_DIR}/runs_phase2", dirs_exist_ok=True)

Copy `DRIVE_DIR/predictions/*.jsonl` into `<meta>/llm/predictions/` of your local meta-dataset and run
`metapilot evaluate` there to get the same table. If an LLM row's `llm_coverage` is well below 1, the model is
not producing complete rankings and its metrics are mostly the single-best fallback; report that, not the row
alone.